In [1]:
import os
import re
import csv
import json
import pickle
import random
import torch
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from copy import deepcopy
from typing import Union, Tuple, Dict
from collections import defaultdict, Counter
from transformers import AutoTokenizer

In [13]:
def pickle_load(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data

def pickle_dump(path, data):
    with open(path, "wb") as f:
        pickle.dump(data, f)

def json_load(path):
    with open(path, mode='r', encoding='utf-8') as f:
        data = json.load(f)
    return data

def json_dump(path, data):
    with open(path, mode='w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

def jsonl_load(path):
    data = []
    with open(path, mode='r', encoding='utf-8') as f:
        # lines = f.readline()
        # lines = json.loads(lines)
        # for line in lines:
        #     data.append(line)
        
        for line in f:
            data.append(json.loads(line))
    return data

In [14]:
data = jsonl_load('../data/QA_Datasets/crag/crag_task_1_and_2_dev_v4.jsonl')
len(data)

2706

In [15]:
qa_list = []
cache = {}
idx = 0
contents2docid = {}

for sample in data:
    qid = sample['interaction_id']
    question = sample['query']
    answer = str(sample['answer'])
    question_type = sample['question_type']
    search_results = sample['search_results']
    split = sample['split']
    if split != 1:
        continue
    if len(answer.split()) >= 4:
        continue

    docs = []
    for doc in search_results:
        contents = doc['page_snippet']
        if len(contents) == 0:
            continue
        docid = contents2docid[contents] if contents in contents2docid else idx
        if docid == idx:
            idx += 1
        docs.append({'id':docid, 'contents':contents})

    if len(docs) == 0:
        continue
    
    cache[question] = docs
    
    qa_list.append({
        'id': qid, 
        'Question': question, 
        'answer': [answer], 
        'type': question_type, 
    })

len(qa_list), len(cache)

(832, 832)

In [16]:
type2qids = defaultdict(list)
for qa in qa_list:
    qid = qa['id']
    question_type = qa['type']
    type2qids[question_type].append(qid)

for k, v in type2qids.items():
    print(f'{k}: {len(v)}')

simple: 315
post-processing: 32
simple_w_condition: 97
aggregation: 96
comparison: 89
multi-hop: 45
false_premise: 152
set: 6


In [17]:
path = '/data/Search-o1/data/QA_Datasets/crag'
json_dump(os.path.join(path, 'crag_full.json'), qa_list)

path = '/data/Search-o1/cache/crag/'
json_dump(os.path.join(path, 'search_cache_full.json'), cache)

In [18]:
random.seed(42)
random.shuffle(qa_list)
sampled_qa_list = qa_list[:500]
sampled_cache = {}
for qa in sampled_qa_list:
    question = qa['Question']
    sampled_cache[question] = deepcopy(cache[question])
len(sampled_qa_list), len(sampled_cache)

(500, 500)

In [19]:
type2qids = defaultdict(list)
for qa in sampled_qa_list:
    qid = qa['id']
    question_type = qa['type']
    type2qids[question_type].append(qid)

for k, v in type2qids.items():
    print(f'{k}: {len(v)}')

simple: 195
comparison: 57
multi-hop: 28
aggregation: 51
false_premise: 94
simple_w_condition: 50
post-processing: 20
set: 5


In [20]:
path = '/data/Search-o1/data/QA_Datasets/crag'
json_dump(os.path.join(path, 'crag_500.json'), sampled_qa_list)

path = '/data/Search-o1/cache/crag/'
json_dump(os.path.join(path, 'search_cache_500.json'), sampled_cache)